# Solution — Multi-API Kafka Pipeline: Producers

Two producers in this notebook:
- **Weather Producer** — polls Open-Meteo every 30 seconds, one message per city → `weather` topic
- **Crypto Producer** — polls CoinGecko every 60 seconds, one message per coin → `crypto` topic

> No API key required for either API.

---
## Shared Setup

In [ ]:
from kafka import KafkaProducer
from datetime import datetime
from time import sleep
import requests
import json

BROKERS = ['course-kafka:9092']

---
## Producer 1 — Weather (`weather` topic)

Uses the [Open-Meteo API](https://open-meteo.com) — completely free, no API key needed.

In [ ]:
WEATHER_TOPIC = 'weather'

CITIES = [
    {"name": "Tel Aviv",  "lat": 32.07,  "lon": 34.78},
    {"name": "London",    "lat": 51.51,  "lon": -0.13},
    {"name": "New York",  "lat": 40.71,  "lon": -74.01},
]

# Map Open-Meteo numeric weather codes to human-readable strings.
# Full list: https://open-meteo.com/en/docs#weathervariables
WEATHER_CODES = {
    0:  "Clear sky",
    1:  "Mainly clear",
    2:  "Partly cloudy",
    3:  "Overcast",
    61: "Rain",
    80: "Showers"
}

In [ ]:
weather_producer = KafkaProducer(
    bootstrap_servers = BROKERS,
    acks              = 1,
    retries           = 3
)

In [ ]:
while True:
    for city in CITIES:

        # Fetch current weather from Open-Meteo — no API key required.
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude":        city["lat"],
                "longitude":       city["lon"],
                "current_weather": True,
                "wind_speed_unit": "kmh"
            },
            timeout=10
        )
        cw = response.json()["current_weather"]

        # Build the enriched payload.
        # WEATHER_CODES.get(code, "Unknown") handles codes not in our lookup table gracefully.
        payload = {
            "city":        city["name"],
            "latitude":    city["lat"],
            "longitude":   city["lon"],
            "temperature": cw["temperature"],
            "windspeed":   cw["windspeed"],
            "condition":   WEATHER_CODES.get(cw["weathercode"], "Unknown"),
            "timestamp":   datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        # Use city name as key so messages for the same city always land
        # in the same partition — useful for per-city ordered processing downstream.
        weather_producer.send(
            topic = WEATHER_TOPIC,
            key   = city["name"].encode("utf-8"),
            value = json.dumps(payload).encode("utf-8")
        )
        weather_producer.flush()

        print(f"Sent → {city['name']} | {payload['temperature']}°C | {payload['condition']}")

    print(f"--- cycle done, sleeping 30s ---")
    sleep(30)

---
## Producer 2 — Crypto (`crypto` topic)

Uses the [CoinGecko API](https://www.coingecko.com/api) — free tier, no API key needed.

In [ ]:
CRYPTO_TOPIC = 'crypto'

COINS = ["bitcoin", "ethereum", "solana"]

In [ ]:
crypto_producer = KafkaProducer(
    bootstrap_servers = BROKERS,
    acks              = 1,
    retries           = 3
)

In [ ]:
while True:

    # One API call returns prices for all coins at once — more efficient than one call per coin.
    response = requests.get(
        "https://api.coingecko.com/api/v3/simple/price",
        params={"ids": ",".join(COINS), "vs_currencies": "usd"},
        timeout=10
    )
    prices = response.json()
    # prices = { "bitcoin": {"usd": 67450.12}, "ethereum": {"usd": 3521.88}, ... }

    for coin, data in prices.items():
        payload = {
            "coin":      coin,
            "price_usd": data["usd"],
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        # Use coin name as key — ensures all messages for a given coin
        # are ordered in a single partition, which matters for price change calculation.
        crypto_producer.send(
            topic = CRYPTO_TOPIC,
            key   = coin.encode("utf-8"),
            value = json.dumps(payload).encode("utf-8")
        )
        crypto_producer.flush()

        print(f"Sent → {coin}: ${data['usd']:,.2f}")

    print(f"--- cycle done, sleeping 60s ---")
    sleep(60)